# VAE Pretraining
Here we will pretrain a VAE using an architecture very similar to the Gómez-Bombarelli / `molecular-vae` style (conv encoder + GRU decoder). The key change is that we will now use SELFIES instead of SMILES.

We will train 3 models from different data sources: ChemBL, Zinc, and ChemBL + Zinc.


### Imports and config

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import selfies as sf

try:
    import wandb
except ImportError:
    wandb = None

SEED = 42
MAX_LEN = 120  # max SELFIES token length before EOS
VAL_FRAC = 0.10
TEST_FRAC = 0.10

LATENT_DIM = 292
EPOCHS = 20
BATCH_SIZE = 128
LR = 1e-3
KL_ANNEAL_EPOCHS = 10
FREE_BITS_NATS = 0.1

USE_WANDB = True
WANDB_PROJECT = "ai-for-toxicology"
WANDB_RUN_NAME = "pretrain-chembl-zinc-seqconv"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
print("device:", device)
print("torch:", torch.__version__)
print("selfies:", sf.__version__)
print("wandb:", "available" if wandb is not None else "not installed (optional)")


device: mps
torch: 2.10.0
selfies: 2.1.1
wandb: available


### Load ChemBL and Zinc datasets

In [2]:
DATA_ROOT = Path("data")
CHEMBL_PATH = DATA_ROOT / "Train" / "chembl_clean.csv"
ZINC_PATH = DATA_ROOT / "Train" / "zinc250k_clean.csv"

for p in [CHEMBL_PATH, ZINC_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")


def load_smiles(path: Path) -> list[str]:
    df = pd.read_csv(path)
    if "canonical_smiles" not in df.columns:
        raise ValueError(f"{path} does not contain canonical_smiles")
    smiles = df["canonical_smiles"].dropna().astype(str).tolist()
    return list(dict.fromkeys(smiles))


chembl_smiles = load_smiles(CHEMBL_PATH)
zinc_smiles = load_smiles(ZINC_PATH)
pretrain_smiles = list(dict.fromkeys(chembl_smiles + zinc_smiles))

print(f"ChemBL unique SMILES: {len(chembl_smiles):,}")
print(f"Zinc unique SMILES:   {len(zinc_smiles):,}")
print(f"Combined unique SMILES for pretraining: {len(pretrain_smiles):,}")


ChemBL unique SMILES: 23,871
Zinc unique SMILES:   249,455
Combined unique SMILES for pretraining: 273,321


### Convert to SELFIES and create the combined pretraining split

In [3]:
def smiles_to_selfies(smiles_list: list[str]) -> tuple[list[str], int]:
    out = []
    failed = 0
    for smi in smiles_list:
        try:
            out.append(sf.encoder(smi))
        except Exception:
            failed += 1
    return out, failed


def filter_selfies_len(selfies_list: list[str], max_len: int = MAX_LEN) -> list[str]:
    return [s for s in selfies_list if len(list(sf.split_selfies(s))) <= max_len]


def split_list(items: list[str], val_frac: float, test_frac: float, seed: int) -> tuple[list[str], list[str], list[str]]:
    rng = np.random.default_rng(seed)
    idx = np.arange(len(items))
    rng.shuffle(idx)

    n = len(items)
    n_test = max(1, int(round(n * test_frac)))
    n_val = max(1, int(round(n * val_frac)))

    if n_test + n_val >= n:
        n_test = max(1, n // 10)
        n_val = max(1, n // 10)

    test_idx = idx[:n_test]
    val_idx = idx[n_test:n_test + n_val]
    train_idx = idx[n_test + n_val:]

    train = [items[i] for i in train_idx]
    val = [items[i] for i in val_idx]
    test = [items[i] for i in test_idx]
    return train, val, test


pretrain_selfies, pretrain_failed = smiles_to_selfies(pretrain_smiles)
pretrain_selfies = filter_selfies_len(pretrain_selfies, max_len=MAX_LEN)
train_selfies, val_selfies, test_selfies = split_list(pretrain_selfies, VAL_FRAC, TEST_FRAC, seed=SEED)

print(f"SELFIES conversion failures: {pretrain_failed}")
print(
    f"filtered combined split sizes: train={len(train_selfies):,}, "
    f"val={len(val_selfies):,}, test={len(test_selfies):,}"
)


SELFIES conversion failures: 0
filtered combined split sizes: train=218,336, val=27,292, test=27,292


### Tokenization and encoding

In [4]:
PAD = "<PAD>"
UNK = "<UNK>"
EOS = "<EOS>"


def tokenize_selfies(s: str) -> list[str]:
    return list(sf.split_selfies(s))


train_tokens = [tokenize_selfies(s) for s in train_selfies]
vocab_tokens = sorted({tok for seq in train_tokens for tok in seq})
ALL_TOKENS = [PAD, UNK, EOS] + vocab_tokens
TOKEN_TO_IDX = {tok: i for i, tok in enumerate(ALL_TOKENS)}
IDX_TO_TOKEN = {i: tok for tok, i in TOKEN_TO_IDX.items()}

PAD_IDX = TOKEN_TO_IDX[PAD]
UNK_IDX = TOKEN_TO_IDX[UNK]
EOS_IDX = TOKEN_TO_IDX[EOS]

SEQ_LEN = MAX_LEN + 1
VOCAB_SIZE = len(ALL_TOKENS)

if SEQ_LEN < 29:
    raise ValueError("Sequence length is too small for conv kernels (9, 9, 11).")


def encode_selfies(s: str) -> list[int]:
    ids = [TOKEN_TO_IDX.get(tok, UNK_IDX) for tok in tokenize_selfies(s)]
    ids = ids[:MAX_LEN]
    ids.append(EOS_IDX)
    return ids


def encode_list(selfies_list: list[str]) -> np.ndarray:
    out = np.full((len(selfies_list), SEQ_LEN), PAD_IDX, dtype=np.int64)
    for i, s in enumerate(selfies_list):
        ids = encode_selfies(s)
        out[i, :len(ids)] = ids
    return out


train_x = encode_list(train_selfies)
val_x = encode_list(val_selfies)
test_x = encode_list(test_selfies)

print(f"train={train_x.shape}, val={val_x.shape}, test={test_x.shape}")
print(f"VOCAB_SIZE={VOCAB_SIZE}, SEQ_LEN={SEQ_LEN}")


train=(218336, 121), val=(27292, 121), test=(27292, 121)
VOCAB_SIZE=115, SEQ_LEN=121


### SELFIES VAE

In [ ]:
class TokenDataset(Dataset):
    def __init__(self, x: np.ndarray):
        self.x = torch.from_numpy(x).long()

    def __len__(self):
        return self.x.size(0)

    def __getitem__(self, idx):
        return self.x[idx]


class PaperLikeSelfiesVAE(nn.Module):
    def __init__(self, vocab_size: int, seq_len: int, latent_dim: int = 292):
        super().__init__()
        self.vocab_size = vocab_size
        self.seq_len = seq_len

        # Conv1d expects [batch, channels, length], so we encode one-hot SELFIES as [B, vocab, seq].
        self.conv_1 = nn.Conv1d(in_channels=vocab_size, out_channels=9, kernel_size=9)
        self.conv_2 = nn.Conv1d(in_channels=9, out_channels=9, kernel_size=9)
        self.conv_3 = nn.Conv1d(in_channels=9, out_channels=10, kernel_size=11)
        self.relu = nn.ReLU()

        with torch.no_grad():
            dummy = torch.zeros(1, vocab_size, seq_len)
            d = self.relu(self.conv_1(dummy))
            d = self.relu(self.conv_2(d))
            d = self.relu(self.conv_3(d))
            flat = d.flatten(1).size(1)

        self.linear_0 = nn.Linear(flat, 435)
        self.linear_1 = nn.Linear(435, latent_dim)
        self.linear_2 = nn.Linear(435, latent_dim)

        self.linear_3 = nn.Linear(latent_dim, 292)
        self.gru = nn.GRU(input_size=292, hidden_size=501, num_layers=3, batch_first=True)
        self.linear_4 = nn.Linear(501, vocab_size)

    def encoder(self, x_onehot: torch.Tensor):
        x = self.relu(self.conv_1(x_onehot))
        x = self.relu(self.conv_2(x))
        x = self.relu(self.conv_3(x))
        x = x.flatten(1)
        x = F.selu(self.linear_0(x))
        return self.linear_1(x), self.linear_2(x)

    def sampling(self, mean: torch.Tensor, logvar: torch.Tensor):
        eps = 1e-2 * torch.randn_like(logvar)
        return torch.exp(0.5 * logvar) * eps + mean

    def decode(self, z: torch.Tensor):
        z = F.selu(self.linear_3(z))
        z_seq = z.unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.gru(z_seq)
        logits = self.linear_4(out)
        return logits

    def forward(self, x_onehot: torch.Tensor):
        mean, logvar = self.encoder(x_onehot)
        z = self.sampling(mean, logvar)
        logits = self.decode(z)
        return logits, mean, logvar


def ids_to_onehot(x_ids: torch.Tensor, vocab_size: int):
    return F.one_hot(x_ids, num_classes=vocab_size).float().transpose(1, 2).contiguous()


def vae_loss(
    logits: torch.Tensor,
    x_ids: torch.Tensor,
    mean: torch.Tensor,
    logvar: torch.Tensor,
    *,
    pad_idx: int,
    beta: float = 1.0,
    free_bits_nats: float = 0.0,
):
    vocab_size = logits.size(-1)
    recon_sum = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        x_ids.reshape(-1),
        ignore_index=pad_idx,
        reduction="sum",
    )

    kl_per_dim = -0.5 * (1 + logvar - mean.pow(2) - logvar.exp())
    if free_bits_nats and free_bits_nats > 0:
        kl_per_dim = torch.clamp(kl_per_dim, min=float(free_bits_nats))
    kl_sum = kl_per_dim.sum()

    n_nonpad = (x_ids != pad_idx).sum().clamp(min=1)
    total = recon_sum + beta * kl_sum
    return total, recon_sum, kl_sum, n_nonpad


def kl_beta(epoch: int, anneal_epochs: int) -> float:
    if anneal_epochs <= 1:
        return 1.0
    return float(min(1.0, epoch / anneal_epochs))


### Training and evaluation helpers

In [6]:
def make_loader(x: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(TokenDataset(x), batch_size=batch_size, shuffle=shuffle)


def init_wandb():
    if not USE_WANDB:
        return None
    if wandb is None:
        raise ImportError("wandb is not installed. Install it or set USE_WANDB=False.")

    return wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        config={
            "seed": SEED,
            "max_len": MAX_LEN,
            "latent_dim": LATENT_DIM,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "kl_anneal_epochs": KL_ANNEAL_EPOCHS,
            "free_bits_nats": FREE_BITS_NATS,
            "vocab_size": VOCAB_SIZE,
            "seq_len": SEQ_LEN,
        },
    )


def run_epoch(model: nn.Module, loader: DataLoader, *, optimizer=None, beta: float = 1.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_sum = 0.0
    recon_sum = 0.0
    kl_sum = 0.0
    n_samples = 0
    n_nonpad = 0
    n_correct = 0

    for x_ids in loader:
        x_ids = x_ids.to(device)
        x_onehot = ids_to_onehot(x_ids, VOCAB_SIZE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits, mean, logvar = model(x_onehot)
            total, recon, kl, nonpad = vae_loss(
                logits,
                x_ids,
                mean,
                logvar,
                pad_idx=PAD_IDX,
                beta=beta,
                free_bits_nats=FREE_BITS_NATS,
            )
            if is_train:
                total.backward()
                optimizer.step()

        mask = x_ids != PAD_IDX
        preds = logits.argmax(dim=-1)

        total_sum += total.item()
        recon_sum += recon.item()
        kl_sum += kl.item()
        n_samples += x_ids.size(0)
        n_nonpad += int(nonpad.item())
        n_correct += ((preds == x_ids) & mask).sum().item()

    return {
        "total": total_sum / max(n_samples, 1),
        "recon_per_token": recon_sum / max(n_nonpad, 1),
        "kl": kl_sum / max(n_samples, 1),
        "token_acc": n_correct / max(n_nonpad, 1),
    }


def evaluate(model: nn.Module, x: np.ndarray, *, beta: float):
    loader = make_loader(x, batch_size=BATCH_SIZE, shuffle=False)
    return run_epoch(model, loader, optimizer=None, beta=beta)


def train_model(train_x: np.ndarray, val_x: np.ndarray):
    model = PaperLikeSelfiesVAE(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN, latent_dim=LATENT_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    train_loader = make_loader(train_x, batch_size=BATCH_SIZE, shuffle=True)
    wandb_run = init_wandb()

    history = {
        "beta": [],
        "train_total": [],
        "val_total": [],
        "train_recon_per_token": [],
        "val_recon_per_token": [],
        "train_kl": [],
        "val_kl": [],
        "train_token_acc": [],
        "val_token_acc": [],
    }

    for ep in range(1, EPOCHS + 1):
        beta = kl_beta(ep, KL_ANNEAL_EPOCHS)
        train_metrics = run_epoch(model, train_loader, optimizer=optimizer, beta=beta)
        val_metrics = evaluate(model, val_x, beta=beta)

        history["beta"].append(beta)
        history["train_total"].append(train_metrics["total"])
        history["val_total"].append(val_metrics["total"])
        history["train_recon_per_token"].append(train_metrics["recon_per_token"])
        history["val_recon_per_token"].append(val_metrics["recon_per_token"])
        history["train_kl"].append(train_metrics["kl"])
        history["val_kl"].append(val_metrics["kl"])
        history["train_token_acc"].append(train_metrics["token_acc"])
        history["val_token_acc"].append(val_metrics["token_acc"])

        print(
            f"epoch {ep:03d} | beta={beta:.2f} | "
            f"train total={train_metrics['total']:.4f} | val total={val_metrics['total']:.4f} | "
            f"train token acc={train_metrics['token_acc']:.4f} | val token acc={val_metrics['token_acc']:.4f}"
        )

        if wandb_run is not None:
            wandb_run.log(
                {
                    "epoch": ep,
                    "beta": beta,
                    "train/total": train_metrics["total"],
                    "val/total": val_metrics["total"],
                    "train/recon_per_token": train_metrics["recon_per_token"],
                    "val/recon_per_token": val_metrics["recon_per_token"],
                    "train/kl": train_metrics["kl"],
                    "val/kl": val_metrics["kl"],
                    "train/token_acc": train_metrics["token_acc"],
                    "val/token_acc": val_metrics["token_acc"],
                },
                step=ep,
            )

    if wandb_run is not None:
        wandb_run.finish()

    return model, history


### Train the ChemBL + Zinc pretraining model

In [ ]:
save_dir = Path("artifacts") / "pretraining_checkpoints"
save_dir.mkdir(parents=True, exist_ok=True)

print("Training on the combined ChemBL + Zinc pretraining set...")
model, history = train_model(train_x, val_x)

final_beta = history["beta"][-1]
test_metrics = evaluate(model, test_x, beta=final_beta)

ckpt_path = save_dir / "paper_like_selfies_chembl_zinc_seqconv_ce.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "token_to_idx": TOKEN_TO_IDX,
        "seq_len": SEQ_LEN,
        "vocab_size": VOCAB_SIZE,
        "max_len": MAX_LEN,
        "pad_idx": PAD_IDX,
        "unk_idx": UNK_IDX,
        "eos_idx": EOS_IDX,
        "history": history,
        "test_metrics": test_metrics,
        "encoder_layout": "onehot_channels_first_seqconv",
        "decoder_output": "logits",
        "loss_name": "token_cross_entropy_plus_kl",
    },
    ckpt_path,
)

results_df = pd.DataFrame(
    [
        {
            "run_name": "chembl_zinc",
            "n_train": len(train_x),
            "n_val": len(val_x),
            "n_test": len(test_x),
            "final_train_total": history["train_total"][-1],
            "final_val_total": history["val_total"][-1],
            "final_train_token_acc": history["train_token_acc"][-1],
            "final_val_token_acc": history["val_token_acc"][-1],
            "test_total": test_metrics["total"],
            "test_recon_per_token": test_metrics["recon_per_token"],
            "test_kl": test_metrics["kl"],
            "test_token_acc": test_metrics["token_acc"],
            "checkpoint": str(ckpt_path),
        }
    ]
)
results_df


Training on the combined ChemBL + Zinc pretraining set...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/iamthomaspruyn/.netrc.
wandb: Currently logged in as: tom-pruyn (tom-pruyn-university-of-toronto) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


epoch 001 | beta=0.10 | train total=90.0162 | val total=86.0805 | train token acc=0.3916 | val token acc=0.4076
epoch 002 | beta=0.20 | train total=86.3504 | val total=82.2724 | train token acc=0.4212 | val token acc=0.4403
epoch 003 | beta=0.30 | train total=84.1833 | val total=81.1066 | train token acc=0.4318 | val token acc=0.4491
epoch 004 | beta=0.40 | train total=81.3111 | val total=81.1959 | train token acc=0.4563 | val token acc=0.4452
epoch 005 | beta=0.50 | train total=78.8462 | val total=75.8712 | train token acc=0.4937 | val token acc=0.5205
epoch 006 | beta=0.60 | train total=76.2904 | val total=72.9227 | train token acc=0.5436 | val token acc=0.5744
epoch 007 | beta=0.70 | train total=74.0415 | val total=73.3436 | train token acc=0.5862 | val token acc=0.5871
epoch 008 | beta=0.80 | train total=72.8538 | val total=74.1479 | train token acc=0.6175 | val token acc=0.5968
epoch 009 | beta=0.90 | train total=72.4203 | val total=69.9440 | train token acc=0.6443 | val token acc

In [ ]:
def evaluate_old_style(model: nn.Module, x: np.ndarray):
    loader = make_loader(x, batch_size=BATCH_SIZE, shuffle=False)
    model.eval()

    total_sum, recon_sum, kl_sum, n = 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for x_ids in loader:
            x_ids = x_ids.to(device)
            x_onehot_seq_vocab = F.one_hot(x_ids, num_classes=VOCAB_SIZE).float()
            x_onehot_vocab_seq = x_onehot_seq_vocab.transpose(1, 2).contiguous()

            logits, mean, logvar = model(x_onehot_vocab_seq)
            probs = F.softmax(logits, dim=-1)

            recon = F.binary_cross_entropy(probs, x_onehot_seq_vocab, reduction="sum")
            kl = -0.5 * torch.sum(1 + logvar - mean.pow(2) - logvar.exp())
            total = recon + kl

            b = x_ids.size(0)
            total_sum += total.item()
            recon_sum += recon.item()
            kl_sum += kl.item()
            n += b

    return {
        "old_total": total_sum / n,
        "old_recon": recon_sum / n,
        "old_kl": kl_sum / n,
    }


In [ ]:
old_style_val = evaluate_old_style(model, val_x)
old_style_test = evaluate_old_style(model, test_x)

print("old-style val:", old_style_val)
print("old-style test:", old_style_test)


### Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = np.arange(1, len(history["train_total"]) + 1)

axes[0].plot(epochs, history["train_total"], label="train")
axes[0].plot(epochs, history["val_total"], label="val")
axes[0].set_title("Total loss / sample")
axes[0].set_xlabel("epoch")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(epochs, history["train_recon_per_token"], label="train")
axes[1].plot(epochs, history["val_recon_per_token"], label="val")
axes[1].set_title("Reconstruction CE / token")
axes[1].set_xlabel("epoch")
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, history["train_token_acc"], label="train")
axes[2].plot(epochs, history["val_token_acc"], label="val")
axes[2].set_title("Token accuracy")
axes[2].set_xlabel("epoch")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Quick reconstruction check

In [ ]:
def decode_ids_to_selfies(ids: np.ndarray | list[int]) -> str:
    toks = []
    for idx in ids:
        tok = IDX_TO_TOKEN[int(idx)]
        if tok == EOS:
            break
        if tok == PAD:
            continue
        toks.append(tok)
    return "".join(toks)


def show_reconstructions(n: int = 5, seed: int = 42):
    model.eval()

    rng = np.random.default_rng(seed)
    k = min(n, len(test_x))
    idxs = rng.choice(len(test_x), size=k, replace=False)

    x_ids = torch.from_numpy(test_x[idxs]).long().to(device)
    x_onehot = ids_to_onehot(x_ids, VOCAB_SIZE)

    with torch.no_grad():
        logits, _, _ = model(x_onehot)
        pred_ids = logits.argmax(dim=-1).cpu().numpy()

    for j, idx in enumerate(idxs):
        orig_selfies = test_selfies[idx]
        pred_selfies = decode_ids_to_selfies(pred_ids[j])
        exact = orig_selfies == pred_selfies
        print(f"[{j}] exact={exact}")
        print("orig:", orig_selfies)
        print("pred:", pred_selfies)
        print()


show_reconstructions(n=5, seed=SEED)


### Notes for later tox21 post-training
This notebook now saves a new sequence-convolution checkpoint. Update the model definition in the post-training notebook before loading it, or keep using the older checkpoint if you want the old architecture.
